# Optimización de Rutas de Entrega (Problema del Viajante)
Este cuaderno interactivo permite experimentar con **distancias y tiempos de entrega** entre puntos logísticos.

Los estudiantes pueden modificar los parámetros y observar **cómo cambia la ruta óptima**.

Conceptos que se trabajan:
- Grafos ponderados
- Optimización de rutas
- Modelación matemática del costo
- Visualización de redes logísticas


In [ ]:
import networkx as nx
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from networkx.algorithms.approximation import traveling_salesman_problem


## Parámetros del modelo
Ajusta las **distancias** entre nodos y los **tiempos de entrega**.

In [ ]:
# Distancias
d_AB = widgets.IntSlider(value=6,min=1,max=20,description='A-B km')
d_AC = widgets.IntSlider(value=8,min=1,max=20,description='A-C km')
d_AD = widgets.IntSlider(value=7,min=1,max=20,description='A-D km')
d_BC = widgets.IntSlider(value=5,min=1,max=20,description='B-C km')
d_BE = widgets.IntSlider(value=9,min=1,max=20,description='B-E km')
d_CD = widgets.IntSlider(value=4,min=1,max=20,description='C-D km')
d_CF = widgets.IntSlider(value=7,min=1,max=20,description='C-F km')
d_DE = widgets.IntSlider(value=6,min=1,max=20,description='D-E km')
d_EF = widgets.IntSlider(value=5,min=1,max=20,description='E-F km')

# Tiempos de entrega
t_B = widgets.IntSlider(value=12,min=5,max=30,description='Tiempo B')
t_C = widgets.IntSlider(value=15,min=5,max=30,description='Tiempo C')
t_D = widgets.IntSlider(value=10,min=5,max=30,description='Tiempo D')
t_E = widgets.IntSlider(value=18,min=5,max=30,description='Tiempo E')
t_F = widgets.IntSlider(value=14,min=5,max=30,description='Tiempo F')

ui = widgets.VBox([
d_AB,d_AC,d_AD,d_BC,d_BE,d_CD,d_CF,d_DE,d_EF,
t_B,t_C,t_D,t_E,t_F
])

display(ui)


## Cálculo y visualización de la ruta óptima

In [ ]:
def actualizar(*args):
    G = nx.Graph()
    nodos = ['A','B','C','D','E','F']
    G.add_nodes_from(nodos)

    tiempos = {
        'A':0,
        'B':t_B.value,
        'C':t_C.value,
        'D':t_D.value,
        'E':t_E.value,
        'F':t_F.value
    }

    aristas = [
        ('A','B',d_AB.value),
        ('A','C',d_AC.value),
        ('A','D',d_AD.value),
        ('B','C',d_BC.value),
        ('B','E',d_BE.value),
        ('C','D',d_CD.value),
        ('C','F',d_CF.value),
        ('D','E',d_DE.value),
        ('E','F',d_EF.value)
    ]

    for u,v,d in aristas:
        costo = d + tiempos[v]
        G.add_edge(u,v,weight=costo)

    ruta = traveling_salesman_problem(G, weight='weight')

    pos = nx.spring_layout(G,seed=3)

    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0,y0 = pos[edge[0]]
        x1,y1 = pos[edge[1]]
        edge_x += [x0,x1,None]
        edge_y += [y0,y1,None]

    edge_trace = go.Scatter(x=edge_x,y=edge_y,mode='lines')

    node_x=[]
    node_y=[]
    for node in G.nodes():
        x,y = pos[node]
        node_x.append(x)
        node_y.append(y)

    node_trace = go.Scatter(x=node_x,y=node_y,mode='markers+text',text=list(G.nodes()),textposition='top center')

    ruta_x=[]
    ruta_y=[]
    for i in range(len(ruta)-1):
        x0,y0 = pos[ruta[i]]
        x1,y1 = pos[ruta[i+1]]
        ruta_x += [x0,x1,None]
        ruta_y += [y0,y1,None]

    ruta_trace = go.Scatter(x=ruta_x,y=ruta_y,mode='lines',line=dict(width=4,color='red'))

    fig = go.Figure(data=[edge_trace,node_trace,ruta_trace])
    fig.update_layout(title='Ruta óptima de entregas')
    fig.show()

for w in [d_AB,d_AC,d_AD,d_BC,d_BE,d_CD,d_CF,d_DE,d_EF,t_B,t_C,t_D,t_E,t_F]:
    w.observe(actualizar,'value')

actualizar()
